# Trisha-v.4.1: Holy Trinity Ensemble
3 model (ConvNeXt V2 Tiny, SwinV2 Small, EVA02 Small 336).
- **5-Fold Stratified Cross-Validation** pada 26.527 data latih
- **Albumentations** augmentasi target: CoarseDropout + ColorJitter + ShiftScaleRotate
- **ConvNeXt V2 Tiny** (28M, 224px) — CNN Edge & Occlusion Master (FCMAE pretrain)
- **SwinV2 Small 256** (50M, 256px) — Transformer Global Context & Shape Master
- **EVA02 Small 336** (22M, 336px) — High-Res Micro-Detail Sniper (2-phase progressive)
- **ClassBalancedLoss** + LabelSmoothing + FocalLoss gamma
- **TTA** (horizontal flip) + Threshold Tuning + Weighted Ensemble
- **Output:** ANGKA 0/1/2 di submission.csv
- **Data:** train/ -> dedup MD5 (no test leakage)

In [1]:
import hashlib
import json
import os
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

%matplotlib inline

_cwd = Path.cwd()
if (_cwd / 'modules').exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd.parent / 'modules').exists():
    sys.path.insert(0, str(_cwd.parent))
else:
    sys.path.insert(0, str(_cwd.parent.parent))

In [2]:
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    print('albumentations OK')
except ImportError:
    print('albumentations not found, install: pip install albumentations')
    A = None

try:
    import bitsandbytes
    print('bitsandbytes OK')
except ImportError:
    print('bitsandbytes not found - will use standard AdamW')
    bitsandbytes = None

albumentations OK
bitsandbytes OK


In [3]:
from modules.utils.config import CLASS_LABELS, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.models.factory import TrashClassifier
from modules.training.loss import ClassBalancedLoss
from modules.training.train import fit, fit_kfold
from modules.ensemble.vote import generate_submission, tune_thresholds, apply_threshold

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = RESULTS / 'trisha_v4.1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Class labels: {CLASS_LABELS}')

d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Output dir: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\experiments\results\trisha_v4.1
Project root: D:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble
Class labels: ['0_Recyclable', '1_Electronic', '2_Organic']


---
## 1. Data Loading + MD5 Dedup

In [ ]:
SRC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
CLEAN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_clean_v4.1'
if not CLEAN_DIR.exists():
    print('Copying train/ to train_clean_v4.1/ ...')
    import shutil
    shutil.copytree(str(SRC_DIR), str(CLEAN_DIR), dirs_exist_ok=True)
    print('Copy complete.')
else:
    print('train_clean_v4.1/ already exists, skipping copy.')
TRAIN_DIR = CLEAN_DIR

def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

long_names = [
    ("1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-visual-representation-of-electronic-waste_73523-11403.jpg", "E_001.jpg"),
    ("12.pile-discarded-motherboards-cpus-cables-disc-drives-hijacked-hardware-heap-concept-electronic-waste_864588-263287.jpg", "E_012.jpg"),
    ("64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat_997534-43151.jpg", "E_064.jpg"),
]
for old_name, new_name in long_names:
    old = TRAIN_DIR / '1_Electronic' / old_name
    new = old.with_name(new_name)
    try:
        os.rename(_safe_path(old), _safe_path(new))
    except (FileNotFoundError, FileExistsError):
        pass
print('Rename long paths done.')

hash_map = defaultdict(list)
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        try:
            with open(_safe_path(fpath), 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})
        except:
            pass

dup_groups = {h: files for h, files in hash_map.items() if len(files) > 1}
removed = 0
for h, files in dup_groups.items():
    for f in files[1:]:
        os.remove(f['path'])
        removed += 1
print(f'Removed {removed} duplicate files.')

records = []
for label_dir in sorted(TRAIN_DIR.iterdir()):
    if not label_dir.is_dir():
        continue
    for img in sorted(label_dir.glob('*')):
        if img.is_file():
            records.append({'path': str(img.relative_to(PROJECT_ROOT)), 'label': label_dir.name})
df = pd.DataFrame(records)
print(f'Total: {len(df)}')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

label_to_idx = {name: i for i, name in enumerate(CLASS_LABELS)}
df['label_idx'] = df['label'].map(label_to_idx)
label_idx_to_name = {i: n for n, i in label_to_idx.items()}

In [7]:
df['path_abs'] = df['path'].apply(lambda p: str(PROJECT_ROOT / p))
df.head(3)

,path,label,label_idx,path_abs
0,data\raw\train\0_Recyclable\R_1.jpg,0_Recyclable,0,D:\Kuliah\LOMBA\Satria-Data\big-data-big-troub...
1,data\raw\train\0_Recyclable\R_10.jpg,0_Recyclable,0,D:\Kuliah\LOMBA\Satria-Data\big-data-big-troub...
2,data\raw\train\0_Recyclable\R_100.jpg,0_Recyclable,0,D:\Kuliah\LOMBA\Satria-Data\big-data-big-troub...


---
## 2. Albumentations Transforms
3 set: ConvNeXtV2 (224px), SwinV2 (256px), EVA02 (224px & 336px).

In [ ]:
if A is None:
    raise ImportError('albumentations required: pip install albumentations')

def make_transform(img_size, is_train=True, extra_blur=False, light_aug=False):
    if is_train:
        ops = [
            A.RandomResizedCrop(img_size, img_size, scale=(0.6, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, p=0.5),
            A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=0.5),
        ]
        if extra_blur:
            ops.append(A.GaussianBlur(blur_limit=(3, 5), p=0.15))
        if light_aug:
            ops = [
                A.RandomResizedCrop(img_size, img_size, scale=(0.7, 1.0)),
                A.HorizontalFlip(p=0.3),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.25),
                A.CoarseDropout(max_holes=4, max_height=16, max_width=16, p=0.25),
            ]
        ops += [ToTensorV2(), A.Normalize(MEAN, STD)]
        return A.Compose(ops)
    return A.Compose([A.Resize(img_size, img_size), ToTensorV2(), A.Normalize(MEAN, STD)])

cnv2_train_tfm = make_transform(224, is_train=True)
cnv2_val_tfm = make_transform(224, is_train=False)

swin_train_tfm = make_transform(256, is_train=True, extra_blur=True)
swin_val_tfm = make_transform(256, is_train=False)

eva_p1_train_tfm = make_transform(224, is_train=True, light_aug=True)
eva_p1_val_tfm = make_transform(224, is_train=False)

eva_p2_train_tfm = make_transform(336, is_train=True, light_aug=True)
eva_p2_val_tfm = make_transform(336, is_train=False)

print('All transforms ready.')

In [ ]:
class TrashDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df['path_abs'].values
        self.labels = df['label_idx'].values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        with open(self.paths[idx], 'rb') as f:
            img = Image.open(BytesIO(f.read())).convert('RGB')
        if self.transform:
            img = self.transform(image=np.array(img))['image']
        return img, self.labels[idx]

def make_loader(subset_df, transform, batch_size=16, shuffle=True):
    ds = TrashDataset(subset_df.reset_index(drop=True), transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

print('TrashDataset ready.')

In [ ]:
def move_to_outdir(name_pattern):
    import shutil
    for f in RESULTS.glob(f'{name_pattern}*'):
        dest = OUT_DIR / f.name
        if not dest.exists():
            shutil.move(str(f), str(dest))

def load_oof(name):
    probs = np.load(OUT_DIR / f'{name}_oof_probs.npy')
    labels = np.load(OUT_DIR / f'{name}_oof_labels.npy')
    return probs, labels

def load_test_models(encoder_name, fold_paths):
    models = []
    for fp in fold_paths:
        m = TrashClassifier(encoder_name=encoder_name, num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(fp, map_location=DEVICE))
        m.to(DEVICE).eval()
        models.append(m)
    return models

@torch.no_grad()
def predict_test(model, test_df, transform, batch_size=32, tta=True):
    model.to(DEVICE).eval()
    ds = TrashDataset(test_df, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    all_probs = []
    for x, _ in tqdm(loader, desc='Predict', leave=False):
        x = x.to(DEVICE)
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        if tta:
            x_flip = torch.flip(x, dims=[3])
            probs = (probs + F.softmax(model(x_flip), dim=1)) / 2
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs)

print('All helpers ready.')

---
## 3. TRAIN: ConvNeXt V2 Tiny (5-Fold)
CNN Edge & Occlusion Master.
- Batch 32, LR 3e-4, CosineAnnealing
- 3 ep head + 7 ep fine-tune = 10 ep/fold
- ClassBalancedLoss + LabelSmoothing(0.1)

In [ ]:
MODEL_CNV2 = 'convnextv2_tiny.fcmae_ft_in22k_in1k'
print(f'=== {MODEL_CNV2} ===')

criterion = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion.update(torch.tensor(df['label_idx'].values))

def cnv2_loaders(train_subset, val_subset):
    return (make_loader(train_subset, cnv2_train_tfm, batch_size=32, shuffle=True),
            make_loader(val_subset, cnv2_val_tfm, batch_size=32, shuffle=False))

mean_f1, oof_p, oof_l, cnv2_ckpts = fit_kfold(
    train_df=df, label_col='label_idx',
    name='convnextv2_tiny', encoder_name=MODEL_CNV2,
    make_loaders_fn=cnv2_loaders,
    accumulation_steps=1, epochs_head=3, epochs_finetune=7,
    lr_head=3e-4, lr_finetune=3e-4, patience=5,
    criterion=criterion, device=DEVICE, use_ema=False, grad_ckpt=False,
)

move_to_outdir('convnextv2_tiny')
print(f'Done. Mean F1: {mean_f1:.4f}')

---
## 4. TRAIN: SwinV2 Small 256 (5-Fold)
Transformer Global Context & Shape Master.
- Batch 16, Grad CKPT ON, LR 1e-4
- 3 ep head + 7 ep fine-tune = 10 ep/fold
- GaussianBlur augmentasi untuk toleransi blur

In [ ]:
MODEL_SWIN = 'swinv2_small_window16_256.ms_in1k'
print(f'=== {MODEL_SWIN} ===')

criterion_s = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion_s.update(torch.tensor(df['label_idx'].values))

def swin_loaders(train_subset, val_subset):
    return (make_loader(train_subset, swin_train_tfm, batch_size=16, shuffle=True),
            make_loader(val_subset, swin_val_tfm, batch_size=16, shuffle=False))

mean_f1_s, oof_p_s, oof_l_s, swin_ckpts = fit_kfold(
    train_df=df, label_col='label_idx',
    name='swinv2_small_256', encoder_name=MODEL_SWIN,
    make_loaders_fn=swin_loaders,
    accumulation_steps=2, epochs_head=3, epochs_finetune=7,
    lr_head=1e-4, lr_finetune=1e-4, patience=5,
    criterion=criterion_s, device=DEVICE, use_ema=False, grad_ckpt=True,
)

move_to_outdir('swinv2_small_256')
print(f'Done. Mean F1: {mean_f1_s:.4f}')

---
## 5. TRAIN: EVA02 Small 336 (5-Fold, 2-Phase)
High-Res Micro-Detail Sniper.
- **Phase 1:** 224px — 8 ep (batch 16, accum 2, LR 1e-4)
- **Phase 2:** 336px — 3 ep (batch 8, accum 4, LR 1e-5, grad_ckpt ON)

In [ ]:
MODEL_EVA = 'eva02_small_patch14_336.mim_in22k_ft_in1k'
print(f'=== {MODEL_EVA} ===')

criterion_e = ClassBalancedLoss(beta=0.999, num_classes=3, label_smoothing=0.1)
criterion_e.update(torch.tensor(df['label_idx'].values))

def eva_p1_loaders(train_subset, val_subset):
    return (make_loader(train_subset, eva_p1_train_tfm, batch_size=16, shuffle=True),
            make_loader(val_subset, eva_p1_val_tfm, batch_size=16, shuffle=False))

def eva_p2_loaders(train_subset, val_subset):
    return (make_loader(train_subset, eva_p2_train_tfm, batch_size=8, shuffle=True),
            make_loader(val_subset, eva_p2_val_tfm, batch_size=8, shuffle=False))

mean_f1_e, oof_p_e, oof_l_e, eva_ckpts = fit_kfold(
    train_df=df, label_col='label_idx',
    name='eva02_small_336', encoder_name=MODEL_EVA,
    make_loaders_fn=eva_p1_loaders,
    phase2_make_loaders_fn=eva_p2_loaders,
    accumulation_steps=2, epochs_head=0, epochs_finetune=8,
    lr_head=0, lr_finetune=1e-4, patience=4,
    phase2_accumulation_steps=4, phase2_lr_finetune=1e-5, phase2_epochs_finetune=3,
    criterion=criterion_e, device=DEVICE, use_ema=False, grad_ckpt=False,
)

move_to_outdir('eva02_small_336')
print(f'Done. Mean F1: {mean_f1_e:.4f}')

---
## 6. Load Test Data
1458 gambar dari `data/raw/test/` (1.jpg s/d 1458.jpg).

In [ ]:
TEST_DIR = PROJECT_ROOT / 'data' / 'raw' / 'test'
test_files = sorted(TEST_DIR.glob('*.jpg'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({
    'path_abs': [str(f) for f in test_files],
    'label_idx': [0] * len(test_files),  # dummy
})
print(f'Test images: {len(test_df)}')

---
## 7. Predict Test — All Models
Load best fold checkpoints, predict dengan TTA, simpan probs.

In [ ]:
def sorted_fold_paths(pattern):
    return sorted(OUT_DIR.glob(pattern),
                  key=lambda p: int(p.stem.split('_fold')[-1].split('_')[0]))

cnv2_folds = sorted_fold_paths('convnextv2_tiny_fold[0-9]*.pt')
swin_folds = sorted_fold_paths('swinv2_small_256_fold[0-9]*.pt')
eva_folds  = sorted_fold_paths('eva02_small_336_fold[0-9]*.pt')

print(f'ConvNeXtV2: {len(cnv2_folds)} folds')
print(f'SwinV2:     {len(swin_folds)} folds')
print(f'EVA02:      {len(eva_folds)} folds')

In [ ]:
print('Predicting ConvNeXt V2 Tiny...')
cnv2_models = load_test_models(MODEL_CNV2, cnv2_folds)
cnv2_test = np.zeros((len(test_df), NUM_CLASSES))
for m in cnv2_models:
    cnv2_test += predict_test(m, test_df, cnv2_val_tfm, batch_size=32, tta=True)
cnv2_test /= len(cnv2_models)
print(f'Done: {cnv2_test.shape}')

In [ ]:
print('Predicting SwinV2 Small 256...')
swin_models = load_test_models(MODEL_SWIN, swin_folds)
swin_test = np.zeros((len(test_df), NUM_CLASSES))
for m in swin_models:
    swin_test += predict_test(m, test_df, swin_val_tfm, batch_size=16, tta=True)
swin_test /= len(swin_models)
print(f'Done: {swin_test.shape}')

In [ ]:
print('Predicting EVA02 Small 336...')
eva_models = load_test_models(MODEL_EVA, eva_folds)
eva_test = np.zeros((len(test_df), NUM_CLASSES))
for m in eva_models:
    eva_test += predict_test(m, test_df, eva_p2_val_tfm, batch_size=8, tta=True)
eva_test /= len(eva_models)
print(f'Done: {eva_test.shape}')

---
## 8. Threshold Tuning
Cari threshold optimal per kelas dari OOF 5-fold.
Ini penting untuk Macro F1 karena kelas minoritas (Electronic) butuh threshold berbeda.

In [ ]:
oof_p_cnv2, oof_l_cnv2 = load_oof('convnextv2_tiny')
oof_p_swin, _ = load_oof('swinv2_small_256')
oof_p_eva, _  = load_oof('eva02_small_336')
true_labels = oof_l_cnv2  # semua model punya OOF labels sama (same KFold split)

blended_oof = (oof_p_cnv2 + oof_p_swin + oof_p_eva) / 3

best_thresh = tune_thresholds(blended_oof, true_labels, n_classes=3, step=0.05)
print(f'Optimal thresholds: {best_thresh}')

pred_before = blended_oof.argmax(axis=1)
pred_after  = apply_threshold(blended_oof, best_thresh)

f1_before = f1_score(true_labels, pred_before, average='macro')
f1_after  = f1_score(true_labels, pred_after, average='macro')
print(f'OOF F1 before threshold: {f1_before:.4f}')
print(f'OOF F1 after threshold:  {f1_after:.4f}')
print(f'Gain: {f1_after - f1_before:+.4f}')

---
## 9. Ensemble + Submission
Blend 0.35 ConvNeXtV2 + 0.35 SwinV2 + 0.30 EVA02. Output angka 0/1/2.

In [ ]:
blended_test = 0.35 * cnv2_test + 0.35 * swin_test + 0.30 * eva_test

test_preds = apply_threshold(blended_test, best_thresh)

unique, counts = np.unique(test_preds, return_counts=True)
print('Test distribution:')
for u, c in zip(unique, counts):
    print(f'  {CLASS_LABELS[u]} ({u}): {c}')

submission = pd.DataFrame({'id': range(1, len(test_preds) + 1), 'predicted': test_preds})
sub_path = OUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path}')
print(submission.head(10).to_string(index=False))

In [ ]:
rows = []
for name in ['convnextv2_tiny', 'swinv2_small_256', 'eva02_small_336']:
    p, _ = load_oof(name)
    f1_val = f1_score(true_labels, p.argmax(axis=1), average='macro')
    rows.append({'model': name, 'oof_f1_macro': f1_val})
rows.append({'model': 'blended (0.35/0.35/0.30)', 'oof_f1_macro': f1_before})
rows.append({'model': 'blended + threshold', 'oof_f1_macro': f1_after})

summary = pd.DataFrame(rows).sort_values('oof_f1_macro', ascending=False)
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f'\nSummary saved to {OUT_DIR / "summary.csv"}')